# 04 — Tumor–stroma interface

The boundary between malignant cells and the surrounding stroma is where most of the interesting biology happens in sarcoma — ECM remodeling, immune exclusion, hypoxia, EMT-like programs. Visium spots at the interface co-express tumor and stromal signatures simply because each spot covers ~10 cells, but we can use that ambiguity to our advantage: we identify the spots that are *neither* pure tumor *nor* pure stroma and look for what's specifically up-regulated there.

We use the cell2location fractions from Notebook 03 to label each spot as one of: `tumor`, `stroma`, `interface`, or `other`, then run differential expression.

In [ ]:
sample           = "GSE227469_angiosarcoma_01"
data_dir         = "data"
results_dir      = "results"
tumor_low        = 0.25
tumor_high       = 0.75
stroma_types     = ["caf", "endothelial", "smooth_muscle"]
tumor_type       = "malignant"

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import scanpy as sc, squidpy as sq
import matplotlib.pyplot as plt

adata = sc.read_h5ad(Path(results_dir) / "visium" / sample / "adata_c2l.h5ad")

## Label spots by compartment

In [ ]:
tumor_frac  = adata.obs[f"fraction_{tumor_type}"]
stroma_frac = sum(adata.obs[f"fraction_{t}"] for t in stroma_types
                  if f"fraction_{t}" in adata.obs.columns)

compartment = np.where(
    (tumor_frac.between(tumor_low, tumor_high)) &
    (stroma_frac.between(tumor_low, tumor_high)), "interface",
    np.where(tumor_frac >= tumor_high, "tumor",
    np.where(stroma_frac >= tumor_high, "stroma", "other"))
)
adata.obs["compartment"] = pd.Categorical(
    compartment, categories=["tumor", "interface", "stroma", "other"]
)
adata.obs["compartment"].value_counts()

In [ ]:
sc.pl.spatial(adata, color="compartment", size=1.4,
              palette={"tumor": "#c44e52", "interface": "#dd8452",
                       "stroma": "#4c72b0", "other": "lightgray"})

## Sanity check — interface spots really do sit at boundaries
Use the spatial neighbor graph to confirm that 'interface' spots tend to neighbor both 'tumor' and 'stroma' spots.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=6)
from scipy.sparse import find
rows, cols, _ = find(adata.obsp["spatial_connectivities"])
neigh = pd.DataFrame({
    "src": adata.obs["compartment"].values[rows],
    "dst": adata.obs["compartment"].values[cols],
})
ctab = pd.crosstab(neigh["src"], neigh["dst"], normalize="index")
ctab

## Differential expression: interface vs. tumor and interface vs. stroma
Run two Wilcoxon tests so we can identify genes that are specifically up at the interface (i.e., not just stromal genes leaking in or tumor genes leaking out).

In [ ]:
sub = adata[adata.obs["compartment"].isin(["tumor", "interface", "stroma"])].copy()

sc.tl.rank_genes_groups(sub, "compartment", method="wilcoxon",
                        groups=["interface"], reference="tumor",
                        key_added="vs_tumor")
sc.tl.rank_genes_groups(sub, "compartment", method="wilcoxon",
                        groups=["interface"], reference="stroma",
                        key_added="vs_stroma")

In [ ]:
vs_tumor  = sc.get.rank_genes_groups_df(sub, group="interface", key="vs_tumor")
vs_stroma = sc.get.rank_genes_groups_df(sub, group="interface", key="vs_stroma")

interface_specific = (
    vs_tumor.merge(vs_stroma, on="names", suffixes=("_vs_tumor", "_vs_stroma"))
            .query("logfoldchanges_vs_tumor > 0.5 and logfoldchanges_vs_stroma > 0.5")
            .sort_values("scores_vs_tumor", ascending=False)
            .head(30)
)
interface_specific

In [ ]:
top_genes = interface_specific["names"].head(9).tolist()
sc.pl.spatial(adata, color=top_genes, ncols=3, cmap="magma", size=1.3)

In [ ]:
out_path = Path(results_dir) / "reports" / f"{sample}_interface_DE.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
interface_specific.to_csv(out_path, index=False)
print("Wrote", out_path)

## Takeaways

- Interface spots reliably neighbor both tumor and stromal spots (see the crosstab above), so the labels are not random — they pick out the actual boundary.
- Genes that beat *both* tumor and stroma reference groups are far more interpretive than a single contrast — they're typically ECM-remodeling, hypoxia, or EMT-related rather than generic tumor or fibroblast markers.
- For sarcoma in particular, expect to see COL/MMP family members, POSTN, FAP, and hypoxia genes (CA9, LOX) over-represented at the interface.